# 04 — Sequences, Dreamer Effects and "Ruling Love": Audit and Re-estimation

MallWorld audit (2026-10)

---

## Scope

This notebook re-derives the claims of:
- *Sequence Motifs in MallWorld Dreams* (archived notebook 06): entries and exits, "loops" and "traps", Markov dynamics and narrative arcs;
- the thesis's variance decomposition ("dreamer identity explains 56.4% of atmosphere variance, location type 7.1%");
- the archived "ruling love" tests RL3 (ICC 0.327) and R4 (partial ρ = 0.007).

The archived sequence notebook cannot run: it reads `reports/data/*.csv`, which do not exist in the repository. Its figures are compared with its last saved outputs.

**Order.** Locations are ordered by their position in the extracted list. Where the coder also gave a `visit_order` for every location of a dream, it agrees with list order in 98.0% of those dreams. The within-dream decline of notebook 02 (C11) is unchanged under list order (checked below).

**Unit.** Location *types* are labels, not places. Two consecutive "other" locations are two different places that share the residual label.

In [1]:
import sys, json, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import chi2_contingency, spearmanr, fisher_exact
from statsmodels.stats.multitest import multipletests

MW_ROOT = Path.cwd().parent
sys.path.insert(0, str(MW_ROOT / "scripts"))
import mallworld_dataset as mw  # verified loader: schema-checked fields, populations, cluster keys

OUTPUT_DIR = MW_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

posts = mw.load_posts()
primary = posts[posts["primary"]].copy()
print(f"Posts: {len(posts):,} | primary dream reports: {len(primary):,} from {primary['author'].nunique():,} authors")

L = mw.load_locations()
L["neg"] = (L["valence"] == -1).astype(float).where(L["valence"].notna())
O = L.sort_values(["post_id", "list_index"]).copy()
O["n_in_dream"] = O.groupby("post_id")["location_id"].transform("size")
seqs = O.groupby("post_id")["location_type"].apply(list)
print(f"Dreams: {len(seqs):,}; with >= 2 locations: {int((seqs.str.len() >= 2).sum()):,}; locations: {len(O):,}")
print(f"'other' is {100 * (O['location_type'] == 'other').mean():.1f}% of location types; distinct names under 'other': {O.loc[O['location_type'] == 'other', 'location_name'].str.lower().nunique():,}")
rng = np.random.default_rng(4)
print("Random 'other' names:", list(O.loc[O["location_type"] == "other", "location_name"].sample(12, random_state=4)))

Posts: 3,732 | primary dream reports: 1,918 from 1,303 authors
Dreams: 1,918; with >= 2 locations: 1,587; locations: 8,685
'other' is 26.4% of location types; distinct names under 'other': 2,217
Random 'other' names: ['road / long water roadway around mountain', 'St Pauls Cathedral', "locked kids' room", 'plain white room (near-death white room)', 'coastal cliff enclosure with house and springs', 'flooded trailer park district', 'rusted/abandoned offshore structure (platform-like)', 'path to the right of a main dorm building (tiled pathway with diagonal arch)', 'elevator with stairs inside', 'Peculiar highway/road shapes', 'desert-like area', 'Huge river with giant rocks (rock-to-rock route)']


## S1 — Entries and exits

**Old.** "Mall locations dominate entries (+8.4% versus exits) while 'other' locations dominate exits (+12.7%)."

In [2]:
S2 = O[O["n_in_dream"] >= 2]
first, last = S2.groupby("post_id").first(), S2.groupby("post_id").last()
rows = []
for t in ["mall", "mall_store", "other", "house", "city_street", "school", "hotel", "parking_lot"]:
    a, b = first["location_type"] == t, last["location_type"] == t
    n_ab, n_ba = int((a & ~b).sum()), int((~a & b).sum())
    rows.append({"type": t, "first %": 100 * a.mean(), "last %": 100 * b.mean(), "first - last (pp)": 100 * (a.mean() - b.mean()),
                 "only first": n_ab, "only last": n_ba, "McNemar exact p": stats.binomtest(n_ab, n_ab + n_ba).pvalue})
ent = pd.DataFrame(rows).set_index("type")
ent["Holm p"] = multipletests(ent["McNemar exact p"], method="holm")[1]
print(f"Dreams with >= 2 locations: {len(first):,}")
with pd.option_context("display.float_format", "{:.3g}".format):
    print(ent.round({"first %": 1, "last %": 1, "first - last (pp)": 1}).to_string())

Dreams with >= 2 locations: 1,587
             first %  last %  first - last (pp)  only first  only last  McNemar exact p   Holm p
type                                                                                            
mall            25.6     5.8               19.8         386         72         5.66e-53 4.53e-52
mall_store       4.7     5.7                 -1          65         81            0.214    0.277
other           15.6    35.9              -20.3         141        463         5.78e-41 4.04e-40
house              6     4.6                1.4          89         67           0.0924    0.277
city_street      6.6     3.3                3.3         101         49         2.63e-05 0.000131
school           4.1     1.6                2.5          60         20         8.58e-06 5.15e-05
hotel            3.6     2.3                1.3          52         31           0.0275     0.11
parking_lot        1     1.8               -0.8          16         28           0.0961    0.

### Findings — S1

The direction of the archived claim holds, and the effect is larger than reported.

| | First location | Last location | Difference |
|---|---|---|---|
| Mall | 25.6% | 5.8% | 19.8 pp |
| "Other" | 15.6% | 35.9% | 20.3 pp |

Both differences have McNemar Holm p < 10⁻³⁹. Basis: 1,587 dreams with at least two locations.

City streets and schools are also more often first than last.

"Other" is the residual label. It covers 26.4% of locations and 2,217 distinct place names, for example a cathedral, a desert, a locked kids' room and an offshore platform. "Dreams end in 'other'" therefore means that dreams end in places the schema did not anticipate. It does not mean they end in one kind of place.

**Status:** descriptive. The pattern is specific to this corpus, whose reports start in the mall by definition. No framework prediction was attached to it.

## S2 — "Loops" and "traps"

**Old.** "43.6% of dreams contained a revisit; 30.2% contained loops; 19.5% were traps with three or more visits to the same location, substantially higher than expected from random walks. 'Other' was the hub of 58.7% of loops and 76.5% of traps."

The archived code counted repeated location **types**: two different shops, both labelled `mall_store`, made a "revisit". No random-walk comparison was computed.

In [3]:
def rep_stats(s):
    reps = s.map(lambda q: len(set(q)) < len(q))
    loops = s.map(lambda q: any(q[i] == q[j] for i in range(len(q)) for j in range(i + 2, len(q))))
    traps = s.map(lambda q: max(Counter(q).values()) >= 3)
    return reps.mean(), loops.mean(), traps.mean()

obs = rep_stats(seqs)
print(f"Archived definitions on the primary population: repeat {100 * obs[0]:.1f}%, 'loop' {100 * obs[1]:.1f}%, 'trap' {100 * obs[2]:.1f}%")
trap_types = Counter(t for q in seqs for t, c in Counter(q).items() if c >= 3)
print(f"Type repeated >= 3 times: 'other' {100 * trap_types['other'] / sum(trap_types.values()):.1f}% of cases; top:", trap_types.most_common(5))
no_other = seqs.map(lambda q: [t for t in q if t != "other"])
o2 = rep_stats(no_other[no_other.str.len() >= 1])
print(f"Ignoring 'other': repeat {100 * o2[0]:.1f}%, 'loop' {100 * o2[1]:.1f}%, 'trap' {100 * o2[2]:.1f}%")

# Null: shuffle type labels across all locations of the corpus, keeping each dream's length
all_types = np.array([t for q in seqs for t in q]); lens = seqs.str.len().values
null = []
for _ in range(200):
    perm = rng.permutation(all_types); out = []; k = 0
    for n_ in lens:
        out.append(list(perm[k:k + n_])); k += n_
    null.append(rep_stats(pd.Series(out)))
null = np.array(null)
for i, name in enumerate(["repeat", "'loop'", "'trap'"]):
    print(f"{name:8s}: observed {100 * obs[i]:.1f}% vs shuffled-label null {100 * null[:, i].mean():.1f}% (95% range {100 * np.percentile(null[:, i], 2.5):.1f}–{100 * np.percentile(null[:, i], 97.5):.1f})")

# Actual returns: a connection whose destination appears earlier in the list than its origin
C = mw.load_connections()
pos = O.set_index(["post_id", "location_id"])["list_index"]
C["from_pos"] = [pos.get((p_, f_), np.nan) for p_, f_ in zip(C["post_id"], C["from_location_id"])]
C["to_pos"] = [pos.get((p_, t_), np.nan) for p_, t_ in zip(C["post_id"], C["to_location_id"])]
back = C.dropna(subset=["from_pos", "to_pos"]).assign(back=lambda d: d["to_pos"] < d["from_pos"])
d_back = back.groupby("post_id")["back"].any()
print(f"Connections back to an earlier-listed location: {int(back['back'].sum())} of {len(back)}; dreams with at least one: "
      f"{int(d_back.sum())} of {len(d_back)} dreams with connections ({100 * d_back.mean():.1f}%)")

Archived definitions on the primary population: repeat 43.7%, 'loop' 30.4%, 'trap' 19.6%
Type repeated >= 3 times: 'other' 66.1% of cases; top: [('other', 287), ('mall_store', 53), ('mall', 22), ('house', 13), ('city_street', 9)]
Ignoring 'other': repeat 24.7%, 'loop' 14.9%, 'trap' 7.2%


repeat  : observed 43.7% vs shuffled-label null 38.5% (95% range 37.2–39.9)
'loop'  : observed 30.4% vs shuffled-label null 28.7% (95% range 27.5–30.0)
'trap'  : observed 19.6% vs shuffled-label null 15.2% (95% range 14.1–16.2)
Connections back to an earlier-listed location: 455 of 4565; dreams with at least one: 340 of 1432 dreams with connections (23.7%)


### Findings — S2

**Reproduced figures.** The archived definitions reproduce on the primary population:
- a repeated type in 43.7% of dreams;
- an A…A "loop" in 30.4%;
- a type repeated three times or more (a "trap") in 19.6%.

**What they measure.**
- These are repeated **labels**, not returns to a place. 66.1% of "traps" are the residual label "other".
- Ignoring "other", the shares fall to 24.7%, 14.9% and 7.2%.
- Against a null that shuffles labels across the corpus, keeping each dream's length, the excess is small:

| | Observed | Shuffled-label null (95% range) |
|---|---|---|
| Repeat | 43.7% | 38.5% (37.2–39.9) |
| Loop | 30.4% | 28.7% (27.5–30.0) |
| Trap | 19.6% | 15.2% (14.1–16.2) |

The claim that these are "substantially higher than expected from random walks" was never computed in the archived notebook. The real excess is 2–5 percentage points.

**Real returns.** The schema records a place only once, so returns are visible only through connections. A connection leads back to an earlier-listed location in 455 of 4,565 connections. At least one such connection occurs in 340 of 1,432 dreams with connections (23.7%).

**Withdrawn:** "loop hub" and "trap" percentages as evidence of entrapment, and "other spaces are where dreamers become most stuck".

## S3 — Markov chain: "'other' is the dominant stationary state (31.7%)"

In [4]:
top = O["location_type"].value_counts()
keep = top[top >= 100].index
lab = O["location_type"].where(O["location_type"].isin(keep), "rare_type")
O["lab"] = lab
pairs = [(a, b) for q in O.groupby("post_id")["lab"].apply(list) for a, b in zip(q[:-1], q[1:])]
states = sorted(set(O["lab"]))
T = pd.crosstab(pd.Series([a for a, _ in pairs], name="from"), pd.Series([b for _, b in pairs], name="to")).reindex(index=states, columns=states, fill_value=0)
P = T.div(T.sum(axis=1), axis=0).values
w, v = np.linalg.eig(P.T)
pi = np.real(v[:, np.argmin(np.abs(w - 1))]); pi = pi / pi.sum()
stat = pd.DataFrame({"stationary %": 100 * pi, "share of all locations %": 100 * O["lab"].value_counts(normalize=True).reindex(states).values,
                     "share of transition targets %": 100 * T.sum(axis=0).values / T.values.sum()}, index=states).sort_values("stationary %", ascending=False)
print(f"Transitions: {len(pairs):,}; states: {len(states)}")
print(stat.round(1).head(8).to_string())
print(f"Max |stationary - share of transition targets| = {np.abs(stat['stationary %'] - stat['share of transition targets %']).max():.2f} pp")
sd = O.groupby("post_id")["lab"].apply(list)
oo = sum(1 for q in sd for a, b in zip(q[:-1], q[1:]) if a == b == "other")
print(f"'other' -> 'other' transitions: {oo} ({100 * oo / len(pairs):.1f}%); expected if labels were independent of position: {100 * (O['lab'] == 'other').mean() ** 2:.1f}%")

Transitions: 6,767; states: 23
             stationary %  share of all locations %  share of transition targets %
other                30.7                      26.4                           29.2
rare_type            21.9                      21.1                           21.9
mall_store            6.2                       6.8                            7.3
mall                  5.7                      10.1                            6.0
city_street           4.1                       4.4                            4.0
house                 3.7                       4.2                            3.7
hotel                 2.9                       2.9                            2.8
school                2.7                       3.0                            2.8
Max |stationary - share of transition targets| = 1.50 pp
'other' -> 'other' transitions: 721 (10.7%); expected if labels were independent of position: 7.0%


### Findings — S3

The stationary share of "other" (30.7% here; 31.7% archived) restates how often "other" is a transition target (29.2%). Over all 23 states, the stationary distribution differs from the target shares by at most 1.5 percentage points. A first-order chain fitted to label bigrams returns the label frequencies, and mean first-passage times are functions of the same matrix.

Consecutive "other" locations are somewhat more common than position-independent labels would give (10.7% of transitions vs 7.0%).

The archived bigram figures (for example 22.3% other→other) cannot be reproduced. Their input CSVs are not in the repository, and the primary-population value is 10.7%.

**Withdrawn:**
- "'other' as universal attractor";
- "mall as gravitational center";
- "parking lots isolated (29–35 steps)".

These describe label frequencies, not dynamics.

## S4 — Atmospheric trajectories and narrative arcs

**Old.**
- "Worsening transitions (529) significantly exceeded improving transitions (456), p = 0.0010." The saved notebook output was 1,080 vs 931; the report's 529 vs 456 appears nowhere.
- "Monotonic trajectories are rare: pure descent 6.7%, pure ascent 5.9%; rise-fall 18.8%, descent-recovery 17.3%".

In [5]:
# Archived transition count: ad hoc 1-8 ordering, consecutive coded locations, no clustering
OLD8 = {"threatening": 1, "oppressive": 2, "eerie": 3, "uncomfortable": 4, "chaotic": 4, "wrong": 3, "neutral": 5, "nostalgic": 6, "welcoming": 7, "peaceful": 8}
A = O[O["atmosphere"] != "not_mentioned"].copy()
A["v8"] = A["atmosphere"].map(OLD8)
dv = A.groupby("post_id")["v8"].apply(lambda s: np.sign(np.diff(s.values)))
imp, wor = int(sum((d > 0).sum() for d in dv)), int(sum((d < 0).sum() for d in dv))
print(f"Archived construction on the primary population: improving {imp}, worsening {wor}; binomial p = {stats.binomtest(imp, imp + wor).pvalue:.3g}")
# Dream as the unit: net direction per dream on the 3-level valence
V = O.dropna(subset=["valence"])
net = V.groupby("post_id")["valence"].apply(lambda s: np.sign(np.diff(s.values)).sum())
net = net[V.groupby("post_id").size() >= 2]
print(f"Dreams with >= 2 coded locations: {len(net)}; net worsening {int((net < 0).sum())}, net improving {int((net > 0).sum())}, "
      f"balanced {int((net == 0).sum())}; sign test p = {stats.binomtest(int((net < 0).sum()), int((net != 0).sum())).pvalue:.3g}")
# Notebook 02 C11 check under list order
V3 = V[V.groupby("post_id")["valence"].transform("size") >= 3].copy()
V3["rel"] = V3.groupby("post_id").cumcount() / (V3.groupby("post_id")["valence"].transform("size") - 1)
sl = V3.groupby("post_id").apply(lambda g: np.polyfit(g["rel"], g["valence"], 1)[0])
print(f"C11 under list order: {len(sl)} dreams, mean slope {sl.mean():+.3f} [{sl.mean() - 1.96 * sl.sem():+.3f}, {sl.mean() + 1.96 * sl.sem():+.3f}], "
      f"declining {100 * (sl < 0).mean():.1f}% vs rising {100 * (sl > 0).mean():.1f}%")

# Narrative arcs: archived classifier and its 1-5 map (which omits eerie, wrong, chaotic and nostalgic)
ARC_MAP = {"threatening": 1, "uncomfortable": 2, "oppressive": 2, "neutral": 3, "peaceful": 4, "welcoming": 5}
def arc(seq):
    n = len(seq)
    f = seq[: n // 3] or seq[:1]; m = seq[n // 3: 2 * n // 3] or seq[1:2]; l_ = seq[2 * n // 3:] or seq[-1:]
    a, b, c = np.mean(f), np.mean(m), np.mean(l_)
    if b < a and b < c: return "descent_recovery"
    if b > a and b > c: return "rise_fall"
    if a > b > c: return "continuous_descent"
    if a < b < c: return "continuous_ascent"
    if abs(a - c) < 0.5: return "stable"
    return "complex"
def arc_table(mapping, label):
    sq = O.assign(v=O["atmosphere"].map(mapping)).dropna(subset=["v"]).groupby("post_id")["v"].apply(list)
    sq = sq[sq.str.len() >= 3]
    obs_ = sq.map(arc).value_counts(normalize=True)
    nulls = []
    for _ in range(500):
        nulls.append(sq.map(lambda q: arc(list(rng.permutation(q)))).value_counts(normalize=True))
    nulls = pd.DataFrame(nulls).fillna(0)
    out = pd.DataFrame({"observed %": 100 * obs_, "shuffled order %": 100 * nulls.mean(),
                        "null 2.5%": 100 * nulls.quantile(0.025), "null 97.5%": 100 * nulls.quantile(0.975)}).fillna(0)
    print(f"{label}: {len(sq)} dreams with >= 3 coded atmospheres")
    print(out.round(1).sort_values("observed %", ascending=False).to_string())
arc_table(ARC_MAP, "Archived map")
arc_table(mw.ATMOSPHERE_OLD_5PT, "All negative atmospheres included (loader's 5-point map)")

Archived construction on the primary population: improving 927, worsening 1075; binomial p = 0.00101


Dreams with >= 2 coded locations: 896; net worsening 175, net improving 130, balanced 591; sign test p = 0.0116


C11 under list order: 594 dreams, mean slope -0.083 [-0.149, -0.018], declining 56.7% vs rising 41.2%


Archived map: 403 dreams with >= 3 coded atmospheres
                    observed %  shuffled order %  null 2.5%  null 97.5%
v                                                                      
complex                   36.0              35.0       31.5        38.2
rise_fall                 17.9              18.8       15.9        22.1
descent_recovery          17.4              18.9       15.6        22.3
stable                    15.9              17.1       16.1        18.4
continuous_descent         7.4               5.0        3.2         6.9
continuous_ascent          5.5               5.1        3.1         7.2


All negative atmospheres included (loader's 5-point map): 608 dreams with >= 3 coded atmospheres
                    observed %  shuffled order %  null 2.5%  null 97.5%
v                                                                      
complex                   36.0              32.5       29.5        35.4
descent_recovery          17.8              19.2       16.1        22.4
stable                    17.1              18.5       17.3        19.7
rise_fall                 15.1              19.5       16.8        22.3
continuous_descent         8.6               5.0        3.5         6.7
continuous_ascent          5.4               5.3        3.8         7.2


### Findings — S4

**Transitions.** The archived construction reproduces: 927 improving vs 1,075 worsening, binomial p = 0.0010. The archived output was 931 vs 1,080.
- That construction treats 2,002 transitions as independent and uses an ad hoc 1–8 ordering.
- The report's "529 vs 456" appears in no output.
- With the dream as the unit, 175 dreams worsen on balance and 130 improve (591 balanced), sign test p = 0.012.
- The within-dream decline of notebook 02 (C11) is unchanged under list order: slope −0.083 (−0.149 to −0.018); 56.7% of dreams decline and 41.2% rise.

**Verdict:** a modest decline is **supported**.

**Arcs.**
- The archived map left out eerie, wrong, chaotic and nostalgic atmospheres, and its "comfortable" is not a schema value.
- Shuffling each dream's atmospheres into random order reproduces the arc distribution almost exactly. Under the archived map, the shares (observed vs shuffled) are:
  - complex 36.0% vs 35.0%;
  - rise–fall 17.9% vs 18.8%;
  - descent–recovery 17.4% vs 18.9%;
  - stable 15.9% vs 17.1%.
- "Oscillatory patterns dominate" and "monotonic trajectories are rare" are therefore what random order produces. They are not narrative structure.
- The one departure is in the other direction: continuous descent is **more** common than chance (7.4% vs a null of 3.2–6.9%; with all negative atmospheres, 8.6% vs 3.5–6.7%), and rise–fall is less common (15.1% vs 16.8–22.3%). This matches the within-dream decline.

**Framework status (added 2026-10-06; data unchanged).** Swedenborg's texts predict no characteristic order of scenes within a dream. His sequence of states is the three states after death in the world of spirits, which some skip (*Heaven and Hell* §491). The arc types above are the archived sequence report's dramatic-structure categories. The absence of a characteristic arc is therefore **not observed**, not a miss of the framework (`CLAUDE.md` §3, "Find the prediction in the text").

## S5 — "Dreamer identity explains 56.4% of atmosphere variance; location type 7.1%"

The archived η² grouped locations by **dream**, not by dreamer, and included dreams with a single coded location. With many small groups, η² is large even when grouping explains nothing: its expected value under no effect is about (groups − 1)/(locations − 1).

In [6]:
V5 = O.assign(v5=O["atmosphere"].map(mw.ATMOSPHERE_OLD_5PT)).dropna(subset=["v5"])
def eta2(df, g, y="v5"):
    m = df.groupby(g)[y].transform("mean"); gm = df[y].mean()
    return ((m - gm) ** 2).sum() / ((df[y] - gm) ** 2).sum()
e_d = eta2(V5, "post_id"); k = V5["post_id"].nunique(); n = len(V5)
perm_d = np.array([eta2(V5.assign(g=rng.permutation(V5["post_id"].values)), "g") for _ in range(200)])
print(f"Dream eta2 = {e_d:.3f} over {n:,} locations in {k:,} dreams; null expectation (k-1)/(n-1) = {(k - 1) / (n - 1):.3f}; permutation null mean {perm_d.mean():.3f}")
tt = V5["location_type"].value_counts(); V5t = V5[V5["location_type"].isin(tt[tt >= 20].index)]
e_t = eta2(V5t, "location_type"); kt = V5t["location_type"].nunique()
perm_t = np.array([eta2(V5t.assign(g=rng.permutation(V5t["location_type"].values)), "g") for _ in range(200)])
print(f"Location-type eta2 = {e_t:.3f} ({kt} types with >= 20 locations); permutation null mean {perm_t.mean():.3f}")
print(f"Chance-corrected: dream {(e_d - perm_d.mean()) / (1 - perm_d.mean()):.3f}; location type {(e_t - perm_t.mean()) / (1 - perm_t.mean()):.3f}")

def icc1(df, group, y):
    g = df.groupby(group)[y]; kk = g.size(); nn = len(df); a = len(kk); grand = df[y].mean()
    msb = (kk * (g.mean() - grand) ** 2).sum() / (a - 1)
    msw = ((df[y] - g.transform("mean")) ** 2).sum() / (nn - a)
    k0 = (nn - (kk ** 2).sum() / nn) / (a - 1)
    return (msb - msw) / (msb + (k0 - 1) * msw)
multi_loc = V5[V5.groupby("post_id")["v5"].transform("size") >= 2]
print(f"Within-dream ICC1 (locations of the same dream; dreams with >= 2 coded locations): {icc1(multi_loc, 'post_id', 'v5'):.3f}")
ds = V5.groupby("post_id").agg(score=("v5", "mean"), author=("author", "first"))
ds = ds[ds["author"].map(ds["author"].value_counts()) >= 2]
print(f"Across-dream author ICC1 (dream means, authors with >= 2 dreams): {icc1(ds, 'author', 'score'):.3f} ({len(ds)} dreams, {ds['author'].nunique()} authors); "
      "compare pre-registered P4 (valence scale): 0.094")

Dream eta2 = 0.544 over 4,293 locations in 1,376 dreams; null expectation (k-1)/(n-1) = 0.320; permutation null mean 0.321


Location-type eta2 = 0.068 (41 types with >= 20 locations); permutation null mean 0.010
Chance-corrected: dream 0.329; location type 0.058
Within-dream ICC1 (locations of the same dream; dreams with >= 2 coded locations): 0.314
Across-dream author ICC1 (dream means, authors with >= 2 dreams): 0.183 (573 dreams, 185 authors); compare pre-registered P4 (valence scale): 0.094


### Findings — S5

**Reproduced figures.** The archived values reproduce: dream η² = 0.544 (archived 0.564) and location-type η² = 0.068 (archived 0.071).

**What they measure.** With 4,293 locations in 1,376 dreams, a grouping that explains nothing gives η² = 0.320 (permutation null 0.321). Each component, corrected for chance:

| Source | Estimate |
|---|---|
| Same dream (within-dream ICC1) | 0.314 (chance-corrected η² 0.329) |
| Same author across different dreams | 0.183 on the 5-point scale (573 dreams, 185 authors); **0.094** on the pre-registered valence scale (P4, a hit) |
| Location type | 0.058 (chance-corrected η²) |

**Verdict.**
- "Dreamer identity explains 56.4%" is **withdrawn**. The figure is a within-dream effect inflated by small groups, and it was labelled as the dreamer.
- The defensible statement is narrower. Atmospheres cluster strongly within a dream. They cluster weakly but reliably within an author's different dreams, and that author-level clustering is larger than the effect of location type.
- **Interpretation:** this ordering fits the framework's claim that a person's ruling love persists and shapes what surrounds them (*Heaven and Hell* §§173–176, 477–479). It also fits stable temperament or writing style (see notebook 07, pre-stated limits).

## S6 — The archived "ruling love" tests

**RL3 (old).** "ICC(1) = 0.327 for congruent fear responses in threatening atmospheres across a dreamer's locations."
- The code selected atmospheres `threatening` or `unsettling`; *unsettling* is not a schema value.
- It counted affects `anxiety`, `fear`, `dread`, `panic` and `horror` as congruent; three of these are not schema values.
- Each location was one observation, so locations of the same dream counted as the same dreamer's independent responses.

**R4 (old).** "Previous affect does not predict the next atmosphere after controlling for the current one (partial ρ = 0.007, p = 0.71): strongly supports the ruling-love model."
- The same cell also printed a second specification, residualized affect, with ρ = 0.053, p = 0.0077. The thesis did not report it.
- A non-significant result was read as positive evidence.

In [7]:
NEG_AFFECT = ["anxiety", "disgust", "horror"]; POS_AFFECT = ["delight", "comfort"]
mw.check_values("affective_response", NEG_AFFECT + POS_AFFECT)
neg_loc = O[O["valence"] == -1].copy()
neg_loc = neg_loc[neg_loc["affective_response"] != "not_mentioned"]
neg_loc["congruent"] = neg_loc["affective_response"].isin(NEG_AFFECT).astype(float)
print(f"Negative-atmosphere locations with a coded affect: {len(neg_loc)} in {neg_loc['post_id'].nunique()} dreams; congruent (anxiety/disgust/horror) {100 * neg_loc['congruent'].mean():.1f}%")
dsc = neg_loc.groupby("post_id").agg(score=("congruent", "mean"), author=("author", "first"))
dsc = dsc[dsc["author"].map(dsc["author"].value_counts()) >= 2]
if dsc["author"].nunique() >= 3:
    obs_icc = icc1(dsc, "author", "score")
    perm = np.array([icc1(dsc.assign(author=rng.permutation(dsc["author"].values)), "author", "score") for _ in range(2000)])
    print(f"RL3 corrected: dream-level ICC1 = {obs_icc:.3f} ({len(dsc)} dreams, {dsc['author'].nunique()} authors); permutation p = {(1 + (perm >= obs_icc).sum()) / 2001:.3f}")

# R4: does affect at one location predict the next location's atmosphere, given the current atmosphere?
O["aff_v"] = np.select([O["affective_response"].isin(NEG_AFFECT), O["affective_response"].isin(POS_AFFECT)], [-1.0, 1.0],
                       np.where(O["affective_response"] == "not_mentioned", np.nan, 0.0))
O["next_neg"] = O.groupby("post_id")["neg"].shift(-1)
R = O.dropna(subset=["aff_v", "valence", "next_neg"]).copy()
R["aff_neg"] = (R["aff_v"] == -1).astype(int); R["cur_neg"] = R["neg"].astype(int)
print(f"R4 pairs (consecutive locations, affect and both atmospheres coded): {len(R)} in {R['post_id'].nunique()} dreams")
print(f"  same-location association, negative affect vs negative atmosphere: phi = {np.corrcoef(R['aff_neg'], R['cur_neg'])[0, 1]:.3f}")
r4 = mw.clustered_logit(R, "next_neg ~ cur_neg + aff_neg + log_words")
print("  next location negative ~ current negative + current negative affect:", mw.odds_ratio_table(r4, ["cur_neg", "aff_neg"]).round(3).to_dict("index"))
from statsmodels.discrete.conditional_models import ConditionalLogit
Rv = R[R.groupby("post_id")["next_neg"].transform("nunique") > 1]
cl = ConditionalLogit(Rv["next_neg"].astype(int), Rv[["cur_neg", "aff_neg"]], groups=Rv["post_id"]).fit(disp=0)
ci = cl.conf_int()
print(f"  within-dream (conditional logit, {Rv['post_id'].nunique()} dreams whose next-location atmosphere varies): "
      + "; ".join(f"{t} OR {np.exp(cl.params[t]):.2f} [{np.exp(ci.loc[t, 0]):.2f}-{np.exp(ci.loc[t, 1]):.2f}], p = {cl.pvalues[t]:.2g}" for t in ["cur_neg", "aff_neg"]))

Negative-atmosphere locations with a coded affect: 2161 in 921 dreams; congruent (anxiety/disgust/horror) 77.6%


RL3 corrected: dream-level ICC1 = 0.041 (353 dreams, 123 authors); permutation p = 0.218
R4 pairs (consecutive locations, affect and both atmospheres coded): 1740 in 681 dreams
  same-location association, negative affect vs negative atmosphere: phi = 0.642
  next location negative ~ current negative + current negative affect: {'cur_neg': {'OR': 3.465, 'lo': 2.555, 'hi': 4.7, 'p': 0.0}, 'aff_neg': {'OR': 2.093, 'lo': 1.591, 'hi': 2.753, 'p': 0.0}}


  within-dream (conditional logit, 183 dreams whose next-location atmosphere varies): cur_neg OR 0.73 [0.49-1.09], p = 0.13; aff_neg OR 1.62 [1.09-2.42], p = 0.017


### Findings — S6

**RL3: withdrawn.**
- On schema values (negative atmosphere; anxiety, disgust or horror as congruent affect), 77.6% of the 2,161 coded negative locations carry congruent affect. That share is close to definitional, because the same sentence usually supplies both codes.
- Consistency across a dreamer's different dreams is ICC1 = 0.041 (353 dreams, 123 authors), permutation p = 0.22. **No evidence** of a stable individual response style.
- The archived 0.327 counted locations of one dream as separate responses of the dreamer, and selected on non-schema values.

**R4: reversed.**
- The data are 1,740 consecutive pairs in 681 dreams.
- Negative affect at one location predicts a negative atmosphere at the next, beyond the current atmosphere: OR 2.09 (1.59–2.75), clustered and length-adjusted.
- Within the same dream (conditional logit, 183 dreams) the OR is 1.62 (1.09–2.42), p = 0.017, while the current atmosphere carries no forward effect (OR 0.73).
- The archived null (ρ = 0.007) came from one specification. Its alternative in the same cell (ρ = 0.053, p = 0.0077) was significant and went unreported.

**Framework fidelity.**
- The archived test assumed that "ruling love" predicts *no* effect of affect on what comes next. It then read a non-significant result as strong support.
- The framework says the opposite about direction: surroundings correspond to, and change with, the state of the person (*Heaven and Hell* §§173–176). Affect leading the next scene is consistent with that.
- This result is exploratory. It is also open to a same-source reading: narrators often signal dread before describing a worse place.

**Status:** **interpretation**; not a test.